###Ingest settlement json file   
1.read file using dataframe reader api   
2.add metadata columns - source file,ingestion timestamp     
3.write bronze tables using dataframe write api

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment_config


In [0]:
%run ../00-common/02.bronze_functions

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/settlement/"

In [0]:
table_name = f"{catalog_name}.{bronze_schema}.settlement"

In [0]:
from pyspark.sql.types import *

In [0]:
settle_schema = StructType([
  StructField('TxnID', StringType()),
  StructField('BillerID', StringType()),
  StructField('CustomerID', StringType()),
  StructField('TxnAmount', DoubleType()),
  StructField('Status', StringType()),
  StructField('SettlementCycleID', StringType()),
  StructField('Timestamp', TimestampNTZType())
])

In [0]:
settle_df = spark.read.format('json') \
                .schema(settle_schema) \
                .option('mode', 'FAILFAST') \
                .option('multiLine', 'true') \
                .load(source_file)

In [0]:
settle_audit = add_ingestion_metadata(settle_df)

In [0]:
settle_final = settle_audit.withColumn("batch_id", F.lit(v_batch_id))

In [0]:
settle_final.write.mode('overwrite').partitionBy('batch_id').option('replaceWhere',f"batch_id = '{v_batch_id}'").saveAsTable(table_name)

In [0]:
display(spark.table(table_name))